In [1]:
from model import UNet
import torch
from torch import nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm


In [2]:
clean_testset_amp  = np.load("data/processed/clean_testset_amp.npy")
clean_trainset_amp = np.load("data/processed/clean_trainset_amp.npy")
noisy_testset_amp  = np.load("data/processed/noisy_testset_amp.npy")
noisy_trainset_amp = np.load("data/processed/noisy_trainset_amp.npy")

In [3]:
torch_clean_testset_amp = torch.from_numpy(clean_testset_amp).float()
torch_clean_trainset_amp = torch.from_numpy(clean_trainset_amp).float()
torch_noisy_testset_amp = torch.from_numpy(noisy_testset_amp).float()
torch_noisy_trainset_amp = torch.from_numpy(noisy_trainset_amp).float()

In [4]:
tensor_test_dataset = TensorDataset(torch_noisy_testset_amp, torch_clean_testset_amp )
tensor_train_dataset = TensorDataset(torch_noisy_trainset_amp, torch_clean_trainset_amp )
train_loader = DataLoader(tensor_train_dataset, batch_size=32, shuffle=True )
test_loader = DataLoader(tensor_test_dataset, batch_size=32)


In [5]:
model = UNet()
device = torch.device("mps")
model = model.to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
num_epochs = 10
best_loss = float('inf')
for epoch in range(num_epochs):
  epoch_loss = 0
  for noisy_batch, clean_batch in tqdm(train_loader):
    optimizer.zero_grad()
    prediction = model(noisy_batch.unsqueeze(1).to(device))
    loss = loss_fn(prediction, clean_batch.unsqueeze(1).to(device))
    epoch_loss += loss.item()
    loss.backward()
    optimizer.step()
  loss_fin = epoch_loss/len(train_loader)
  if best_loss > loss_fin:
      best_loss = loss_fin
      torch.save(model.state_dict(), "models/best_model.pth")
  print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_fin}")
  

100%|██████████| 1239/1239 [07:22<00:00,  2.80it/s]


Epoch 1/10, Loss: 0.21279836569922503


100%|██████████| 1239/1239 [07:21<00:00,  2.81it/s]


Epoch 2/10, Loss: 0.17777437533614998


100%|██████████| 1239/1239 [07:21<00:00,  2.80it/s]


...

Epoch 8/10, Loss: 0.12270886052318693


100%|██████████| 1239/1239 [07:31<00:00,  2.75it/s]


Epoch 9/10, Loss: 0.11600840926362778


100%|██████████| 1239/1239 [07:27<00:00,  2.77it/s]

Epoch 10/10, Loss: 0.11399522912632178
